In [4]:
import librosa
import IPython.display as ipd
import numpy as np
import scipy.fft as fft
from scipy.signal import stft, istft, lfilter
import soundfile as sf

In [5]:
filename = '../../data/tomsdiner_mono.wav'
diner, srate = librosa.load(filename)
ipd.Audio(diner, rate=srate)

## MY WORK

In [6]:
# ==== FFT BIN FILTERING ====
# Filter all but the k highest (or lowest) FFT bins
def filter_high_low_mag_peaks(input, k, highest):
    complex_spectrum = fft.rfft(input)
    magnitude_spectrum = np.abs(complex_spectrum)
    sorted_mag_spectrum = np.sort(magnitude_spectrum)
    if not highest:
        sorted_mag_spectrum = np.flip(sorted_mag_spectrum)
    threshold = sorted_mag_spectrum[-k]
    if highest:
        magnitude_spectrum[magnitude_spectrum < threshold] = 0
    else:
        magnitude_spectrum[magnitude_spectrum > threshold] = 0
    num_bins = len(magnitude_spectrum)
    phase_spectrum = np.angle(complex_spectrum)
    reconstructed_complex_spectrum  = magnitude_spectrum*(np.cos(phase_spectrum)+1j*np.sin(phase_spectrum))
    reconstructed = fft.irfft(reconstructed_complex_spectrum)
    return reconstructed

# Filter all FFT bins between lo and hi
def window_reconstruct(input, lo, hi):
    complex_spectrum = np.fft.rfft(input)
    magnitude_spectrum = np.abs(complex_spectrum)
    sorted_mag_spectrum_indices = np.flip(np.argsort(magnitude_spectrum))
    mask = np.zeros(len(complex_spectrum), bool)
    mask[sorted_mag_spectrum_indices[lo:hi]] = True
    return np.fft.irfft(complex_spectrum * mask, n=len(input))

# Filter FFT bins randomly
def weighted_random_reconstruct(input, k):
    rng = np.random.default_rng()
    complex_spectrum = np.fft.rfft(input)
    magnitude_spectrum = np.abs(complex_spectrum)

    weights = magnitude_spectrum
    weights[0] = 0
    p = weights / weights.sum()

    idx = rng.choice(len(complex_spectrum), size=k, replace=False, p=p)

    mask = np.zeros(len(complex_spectrum), dtype=bool)
    mask[idx] = True
    return np.fft.irfft(complex_spectrum * mask, n=len(input))

# Filter all but the k highest FFT bins for each frame (frame size determined by nperseg)
def stft_topk(input, k, nperseg=2**15):
    f, t, Z = stft(input, srate, nperseg=nperseg)
    mag = np.abs(Z)
    idx = np.argpartition(mag, -k, axis=0)[-k:]
    mask = np.zeros_like(mag, dtype=bool)
    np.put_along_axis(mask, idx, True, axis=0)
    _, y = istft(Z * mask, srate, nperseg=nperseg)
    return y

def filter_mag_peaks(input, lim, strategy='highest', arg2=None):
    if strategy == 'highest' or strategy == 'lowest':
        highest = strategy == 'highest'
        return filter_high_low_mag_peaks(input, lim, highest=highest)
    elif strategy == 'window':
        return window_reconstruct(input, lim, arg2)
    elif strategy == 'random':
        return weighted_random_reconstruct(input, lim)
    elif strategy == 'frames':
        return stft_topk(input, lim, nperseg=arg2)

# ==== KS STRINGS ====
# Synthesize a KS string using a sound clip instead of frequency
def ks_clip_resonator(duration, clip, sample_rate=44100, decay=0.996, normalize=True):
    ring_buffer = np.asarray(clip, dtype=float)
    ring_buffer = ring_buffer - ring_buffer.mean()

    num_samples = int(duration * sample_rate)
    audio = np.zeros(num_samples)
 
    for i in range(num_samples):
        audio[i] = ring_buffer[0]

        new_sample = decay * 0.5 * (ring_buffer[0] + ring_buffer[1])

        ring_buffer = np.roll(ring_buffer, -1)
        ring_buffer[-1] = new_sample
 
    if normalize and np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio))
    return audio

# Synthesize a KS string by passing diner in to constantly excite the string
def ks_filter(x, period, decay=0.996, blend=0.5, invert=False):
    sign = 1.0 if invert else -1.0
    a = np.zeros(period + 2)
    a[0] = 1.0
    a[period] = sign * decay * (1 - blend)
    a[period + 1] = sign * decay * blend
    return lfilter([1.0], a, x)

# Synthesize a KS string using a clip segment as the initial excitation (pluck) of the string
def pluck(excitation, freq, duration, sr, **kw):
    x = np.zeros(int(duration * sr))
    x[:len(excitation)] = excitation - np.mean(excitation)
    y = ks_filter(x, int(round(sr / freq - 0.5)), **kw)
    return y / np.max(np.abs(y))

# Found this online; apply feedback to each sample by delaying 
def ks_nonlinear(input, delay, decay=0.996):
    num_samples = len(input)
    audio = np.zeros(num_samples)
    for i in range(num_samples):
        a = audio[i - delay] if i >= delay else 0.0
        b = audio[i - delay - 1] if i >= delay + 1 else 0.0
        feedback = np.tanh(decay * (0.5 * a + 0.5 * b))
        audio[i] = input[i] + feedback
    return audio

# ==== REVERB ====
# Bounce audio left and right
def ping_pong(x, delay, sr, decay=0.7, h=(0.5, 0.5), tail=3.0):
    D = int(round(delay))
    out = np.zeros((len(x) + int(tail * sr), 2))
    out[:len(x)] += x[:, None]
    tap = x.copy()
    peak = np.max(np.abs(tap)) + 1e-12
    r = 0
    while True:
        r += 1
        tap = decay * lfilter(h, [1.0], tap)
        if np.max(np.abs(tap)) < 1e-4 * peak or r * D >= len(out):
            break
        end = min(len(out), r * D + len(tap))
        out[r * D:end, r % 2] += tap[:end - r * D] # [:, r%2] grabs left and right channels alternating
    return np.transpose(out) # [[1, 2], [3, 4], [5, 6]] -> [[1, 3, 5], [2, 4, 6]]

# helpers
def _rms(a):
    return np.sqrt(np.mean(np.asarray(a) ** 2)) + 1e-12

def _damped_comb(x, delay, rt60, damp, sr):
    fb = 10.0 ** (-3.0 * delay / (sr * rt60))
    a = np.zeros(delay + 2)
    a[0], a[1] = 1.0, -damp
    a[delay] -= fb * (1 - damp)
    b = np.zeros(delay + 2)
    b[delay], b[delay + 1] = 1.0, -damp
    return lfilter(b, a, x)
 
 
def _allpass(x, delay, g=0.5):
    b = np.zeros(delay + 1)
    a = np.zeros(delay + 1)
    b[0], b[delay] = -g, 1.0
    a[0], a[delay] = 1.0, -g
    return lfilter(b, a, x)

def _finish(x, wet, sr, mix, predelay):
    n_out = len(wet)
    if predelay > 0:
        pd = int(predelay * sr)
        wet = np.vstack([np.zeros((pd, 2)), wet])[:n_out]
    wet = wet * (_rms(x) / _rms(wet[:len(x)]))
    dry = np.zeros(n_out)
    dry[:len(x)] = x
    m = np.asarray(mix, float)
    if m.ndim:
        m = np.interp(np.linspace(0, 1, n_out), np.linspace(0, 1, len(m)), m)
        m = m[:, None]
    return (1 - m) * dry[:, None] + m * wet

# Freeverb implementation (https://ccrma.stanford.edu/~jos/pasp/Freeverb.html)
def freeverb(x, sr, decay=2.0, damp=0.3, mix=0.3, predelay=0.0, tail=None, spread=23):
    s = sr / 44100
    tail_n = int((decay if tail is None else tail) * sr)
    xp = np.concatenate([x, np.zeros(tail_n)])
    wet = np.zeros((len(xp), 2))
    comb = [1116, 1188, 1277, 1356, 1422, 1491, 1557, 1617]
    ap = [556, 441, 341, 225]
    for ch in range(2):
        off = ch * int(spread * s)
        acc = sum(_damped_comb(xp, int(d * s) + off, decay, damp, sr) for d in comb)
        for d in ap:
            acc = _allpass(acc, int(d * s) + off)
        wet[:, ch] = acc
    return np.transpose(_finish(x, wet, sr, mix, predelay))

# ==== CONCATENATIVE SYNTHESIS ====
# helpers
def fade(x, ms=5):
    n = int(srate * ms / 1000)
    ramp = np.linspace(0, 1, n)
    x = x.copy()
    x[:n] *= ramp
    x[len(x) - n:] *= ramp[::-1]
    return x

def take(bank, nm):
    count = {}
    count[nm] = count.get(nm, -1) + 1
    return bank[nm][count[nm] % len(bank[nm])]

def layer(a, b, offset=0):
    out = np.zeros(max(len(a), offset + len(b)))
    out[:len(a)] += fade(a)
    out[offset:offset + len(b)] += fade(b)
    return out

def get_concat_mels():
    # get all notes
    segs_8 = np.array_split(diner, 16)
    segs_16 = np.array_split(diner, 32)
    notes = []
    for i in range(5):
        notes.append(segs_8[i])
    notes.append(segs_16[10])
    notes.append(np.hstack((segs_16[11], segs_8[6])))
    for i in range(7, 13):
        notes.append(segs_8[i])
    notes.append(segs_16[26])
    notes.append(np.hstack((segs_16[27], segs_8[14])))
    notes.append(segs_8[15])
    
    names = ['F#', 'G#', 'A', 'C#', 'F#', 'G#', 'A', 'C#', 'F#', 'C#', 'B', 'A', 'B', 'A', 'B', 'A']
    pitch_of = {'F#': 66, 'G#': 68, 'A': 69, 'B': 71, 'C#': 73}
    
    midi = np.array([pitch_of[n] for n in names])
    loud = np.array([np.sqrt(np.mean(n ** 2)) for n in notes])
    
    # bank of notes sorted by loudness
    bank = {}
    for i in np.argsort(loud):
        bank.setdefault(names[i], []).append(notes[i])
    
    composed_tune = [bank[nm][0] for nm in ['B', 'C#', 'A', 'A', 'B', 'C#', 'A', 'A', 'A', 'B', 'G#', 'G#', 'A', 'B', 'G#', 'G#']]
    
    rising = np.hstack([fade(notes[i]) for i in np.argsort(midi)])
    swell = np.hstack([fade(notes[i]) for i in np.argsort(loud)])
    backwards = np.hstack([fade(n) for n in notes[::-1]])
    
    segs_8_shf = np.array_split(diner, 16)
    np.random.shuffle(segs_8_shf)
    random_tune_8 = np.hstack(segs_8_shf)
    
    segs_16_shf = np.array_split(diner, 32)
    np.random.shuffle(segs_16_shf)
    random_tune_16 = np.hstack(segs_16_shf)
    
    feats = np.array([librosa.feature.mfcc(y=n, sr=srate, n_mfcc=13).mean(axis=1) for n in notes])
    dist = np.linalg.norm(feats[:, None] - feats[None], axis=2)
    np.fill_diagonal(dist, np.inf)
    twin = dist.argmin(axis=1)
    swapped_closest = np.hstack([fade(notes[i]) for i in twin])
    
    swapped_instance = np.hstack([fade(take(bank, nm)) for nm in names])
    
    order = ['F#', 'G#', 'A', 'B', 'C#']
    mirror = dict(zip(order, order[::-1]))
    inverted = np.hstack([fade(bank[mirror[nm]][0]) for nm in names])
    
    layered = np.hstack([layer(notes[i], notes[i + 1]) for i in range(0, 16, 2)])
    layered /= np.abs(layered).max()

    return (random_tune_8, random_tune_16, np.hstack(composed_tune), rising, swell, backwards, swapped_closest, swapped_instance, inverted, layered)


random_tune_8, random_tune_16, composed_tune, rising, swell, backwards, swapped_closest, swapped_instance, inverted, layered = get_concat_mels()


## TRINITY'S WORK

In [7]:
# ==== MAGNITUDE SPECTRUM FILTERING ====
# Whereas my FFT bin filtering was based on how loud the bins were, this
# method is focused on where in the spectrum the bins lie.
def crude_filtering(signal: np.ndarray, frequency_range: tuple[float, float]) -> np.ndarray:
  complex_spectrum = fft.rfft(signal)
  magnitude_spectrum = np.abs(complex_spectrum)

  num_bins = len(magnitude_spectrum)

  # like percentages of the freq bins
  start = (frequency_range[0] * num_bins)
  end = (frequency_range[1] * num_bins)
  magnitude_spectrum[int(start):int(end)] = 0.0

  phase_spectrum = np.angle(complex_spectrum)
  reconstructed_complex_spectrum = magnitude_spectrum * (np.cos(phase_spectrum)+1j*np.sin(phase_spectrum))

  reconstructed_signal = fft.irfft(reconstructed_complex_spectrum)

  return reconstructed_signal


def band_pass_crude_filtering(signal: np.ndarray, frequency_range: tuple[float, float]) -> np.ndarray:
  lower_frequency_range = (0.0, frequency_range[0])
  upper_frequency_range = (frequency_range[1], 1.0)

  lower_filtered_signal = crude_filtering(signal, lower_frequency_range)
  band_pass_filtered_signal = crude_filtering(lower_filtered_signal, upper_frequency_range)

  return band_pass_filtered_signal

low_pass = (0.10, 1.00)
high_pass = (0.0, 0.35)
band_stop = (0.250, 0.75)
band_pass = (0.20, 0.70)

# ==== FLAT MAGNITUDE RECONSTRUCTION ====
def flat_magnitude_reconstruct(signal: np.ndarray, amplitude: float) -> np.ndarray:
  complex_spectrum = fft.rfft(signal)
  magnitude_spectrum = np.full(len(complex_spectrum), amplitude)
  phase_spectrum = np.angle(complex_spectrum)
  reconstructed_complex_spectrum = magnitude_spectrum *(np.cos(phase_spectrum) + 1j*np.sin(phase_spectrum))
  reconstructed_signal = fft.irfft(reconstructed_complex_spectrum)

  return reconstructed_signal

## COMBOS

In [ ]:
# 